# MÓDULO 4
## Tema 3. Introducción a bases de datos con SQLite y Python (sqlite3)

**Objetivos:**
- Crear y operar una base de datos SQLite con `sqlite3`.
- Diseñar tablas, insertar/consultar/actualizar/borrar (CRUD).
- Entender transacciones, índices y consultas parametrizadas (seguridad).
- Conectar SQLite con análisis (incluyendo lectura con Pandas).

## Índice
- Qué es SQLite y cuándo usarlo
- SQL básico en 10 minutos (CREATE/SELECT/INSERT/UPDATE/DELETE)
- Tipos de datos, afinidad y restricciones (PRIMARY KEY, NOT NULL, UNIQUE, CHECK, DEFAULT)
- Explorar la base de datos (sqlite_master, PRAGMA y comandos útiles del shell)
- Crear tablas (modelo simple)
- CRUD con consultas parametrizadas (seguridad) y `executemany`
- Modelo relacional realista (claves foráneas) y JOINs
- Transacciones y control de excepciones (`IntegrityError`, rollback)
- Índices y `EXPLAIN QUERY PLAN`
- Integración con Pandas
- Trampas comunes y buenas prácticas
- Mini-práctica


## Qué es SQLite y cuándo usarlo

SQLite es una base de datos embebida (un fichero `.db`), ideal para:
- prototipos
- herramientas internas
- pipelines locales
- pruebas y datasets medianos

Ventajas:
- cero servidor
- estándar SQL
- fiable y muy usada

Limitaciones:
- concurrencia de escritura limitada (no es para cargas masivas multiusuario)

## SQL básico en 10 minutos (lo justo para empezar)

SQLite se maneja con **SQL**. En este tema usaremos principalmente:

| Operación | Qué hace | Ejemplo mínimo |
|---|---|---|
| `CREATE TABLE` | crea una tabla | `CREATE TABLE usuarios(...);` |
| `INSERT INTO` | inserta filas | `INSERT INTO usuarios(email) VALUES (?);` |
| `SELECT` | consulta datos | `SELECT * FROM usuarios;` |
| `UPDATE` | actualiza filas | `UPDATE usuarios SET pais=? WHERE email=?;` |
| `DELETE` | borra filas | `DELETE FROM usuarios WHERE email=?;` |
| `CREATE INDEX` | acelera búsquedas | `CREATE INDEX ... ON usuarios(pais);` |

**Regla de oro con Python:** nunca construyas SQL con f-strings cuando hay datos de usuario. Usa parámetros `?`.


## Tipos de datos en SQLite (afinidad) y cómo elegirlos

SQLite tiene pocos tipos "oficiales", pero usa una idea llamada **type affinity** (afinidad): tú declaras un tipo y SQLite intenta tratarlo como tal.

| Tipo declarado (común) | Úsalo para | Ejemplo |
|---|---|---|
| `INTEGER` | IDs, contadores, flags 0/1 | `id INTEGER PRIMARY KEY` |
| `REAL` | importes, medidas | `total REAL NOT NULL` |
| `TEXT` | emails, estados, fechas ISO | `created_at TEXT NOT NULL` |
| `BLOB` | binarios (raramente en curso) | bytes |
| `NULL` | ausencia de valor | `NULL` |

### Fechas y horas
En SQLite no hay tipo `DATETIME` "real". Lo habitual es:
- **TEXT** en ISO-8601: `2026-01-26T14:30:00` (fácil de leer/depurar)
- **INTEGER** con epoch seconds (útil si haces cálculos de tiempo)

En este módulo usaremos **TEXT ISO** por claridad.


## Restricciones (constraints) que sí merece la pena usar desde el principio

| Restricción | Qué garantiza | Ejemplo |
|---|---|---|
| `PRIMARY KEY` | identificador único por fila | `id INTEGER PRIMARY KEY` |
| `NOT NULL` | el campo es obligatorio | `email TEXT NOT NULL` |
| `UNIQUE` | evita duplicados | `email TEXT UNIQUE` |
| `CHECK` | valida un conjunto/condición | `CHECK(status IN ('paid','pending','cancelled'))` |
| `DEFAULT` | valor por defecto | `created_at TEXT DEFAULT (datetime('now'))` |

> Nota: `INTEGER PRIMARY KEY` en SQLite crea un ID autoincremental (rowid) sin que tengas que hacer nada.


## Explorar la base de datos (ver tablas y esquema)

Cuando no recuerdas qué hay dentro de la base de datos, tienes dos vías:

### 1) Desde SQL (sirve en Python)
- Listar tablas: `SELECT name FROM sqlite_master WHERE type='table';`
- Ver el SQL de creación: `SELECT sql FROM sqlite_master WHERE name='pedidos';`
- Ver columnas: `PRAGMA table_info(pedidos);`
- Ver índices: `PRAGMA index_list('pedidos');`

### 2) Desde el shell `sqlite3` (comandos que empiezan por punto)
Estos comandos **no son SQL** (solo funcionan dentro del terminal `sqlite3`):

```sql
.tables
.schema pedidos
.indexes pedidos
.headers on
.mode column
```


In [1]:
import sqlite3
from pathlib import Path

db_path = Path("tienda.db")
conn = sqlite3.connect(db_path)
conn.close()
print("DB creada:", db_path)

DB creada: tienda.db


## Crear tablas

Caso realista: guardar pedidos y eventos de usuario.

Buenas prácticas mínimas:
- claves primarias
- tipos coherentes
- restricciones simples cuando aplican

In [2]:
import sqlite3
from pathlib import Path

db_path = Path("tienda.db")

with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS pedidos (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            total REAL NOT NULL,
            status TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS eventos (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            tipo TEXT NOT NULL,
            ts TEXT NOT NULL
        );
    """)
print("Tablas creadas")

Tablas creadas


## Inspección práctica desde Python

Ejemplos rápidos para ver qué tablas y columnas existen sin salir de Python.

In [3]:
import sqlite3

def listar_tablas(conn):
    cur = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    return [r[0] for r in cur.fetchall()]

with sqlite3.connect("tienda.db") as conn:
    print("Tablas:", listar_tablas(conn))
    print("\nColumnas de pedidos (PRAGMA table_info):")
    for col in conn.execute("PRAGMA table_info(pedidos);"):
        # (cid, name, type, notnull, dflt_value, pk)
        print(col)

    print("\nSQL de creación de pedidos:")
    row = conn.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='pedidos';").fetchone()
    print(row[0])


Tablas: ['eventos', 'pedidos']

Columnas de pedidos (PRAGMA table_info):
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'email', 'TEXT', 1, None, 0)
(2, 'total', 'REAL', 1, None, 0)
(3, 'status', 'TEXT', 1, None, 0)
(4, 'created_at', 'TEXT', 1, None, 0)

SQL de creación de pedidos:
CREATE TABLE pedidos (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            total REAL NOT NULL,
            status TEXT NOT NULL,
            created_at TEXT NOT NULL
        )


## CRUD con consultas parametrizadas

Regla de oro: **nunca concatenes** inputs de usuario en SQL.

Usa parámetros `?` para evitar SQL injection y problemas de escaping.

In [5]:
import sqlite3
from datetime import datetime

def now_iso():
    return datetime.now().isoformat(timespec="seconds")

with sqlite3.connect("tienda.db") as conn:
    cur = conn.cursor()

    # INSERT
    cur.execute(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?)",
        ("ana@example.com", 19.99, "paid", now_iso()),
    )
    print("INSERT ana -> filas:", cur.rowcount)

    cur.execute(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?)",
        ("luis@example.com", 55.00, "pending", now_iso()),
    )
    print("INSERT luis -> filas:", cur.rowcount)

    # SELECT
    cur.execute("SELECT id, email, total, status FROM pedidos WHERE status = ?", ("paid",))
    print("Paid:", cur.fetchall())

    # UPDATE
    cur.execute("UPDATE pedidos SET status = ? WHERE email = ?", ("paid", "luis@example.com"))
    print("UPDATE luis -> filas:", cur.rowcount)

    # DELETE (ejemplo: borrar pedidos de test)
    cur.execute("DELETE FROM pedidos WHERE email LIKE ?", 
                ("%test%",))
    print("DELETE test -> filas:", cur.rowcount)

INSERT ana -> filas: 1
INSERT luis -> filas: 1
Paid: [(1, 'ana@example.com', 19.99, 'paid'), (2, 'luis@example.com', 55.0, 'paid'), (3, 'ana@example.com', 19.99, 'paid')]
UPDATE luis -> filas: 2
DELETE test -> filas: 0


## Insertar muchos registros: `executemany` y resultados por nombre (`sqlite3.Row`)

Dos trucos muy prácticos:
- `executemany(...)` para insertar listas de filas de forma eficiente.
- `row_factory = sqlite3.Row` para acceder por nombre de columna (más legible que índices).

#### `row_factory` en SQLite

Por defecto, SQLite devuelve cada fila como una **tupla**, por lo que los datos se acceden por posición:

r[0]  → id  
r[1]  → email  
r[2]  → total  

Este enfoque es:
- frágil (si cambia el orden de columnas, el código se rompe)
- poco legible
- difícil de mantener

Pero si configuramos la conexión con: conn.row_factory = sqlite3.Row

cada fila pasa a comportarse como un **diccionario ordenado**, permitiendo acceder a los datos por nombre de columna:

r["email"]  
r["total"]  
r["status"]  

Esto hace el código:
- más claro
- más robusto ante cambios
- especialmente útil en consultas grandes, JOINs y cuando se convierten resultados a JSON

In [6]:
import sqlite3
from datetime import datetime

def now_iso():
    return datetime.now().isoformat(timespec="seconds")

pedidos_extra = [
    ("ana@example.com", 9.99, "paid", now_iso()),
    ("ana@example.com", 3.50, "paid", now_iso()),
    ("luis@example.com", 120.00, "cancelled", now_iso()),
]

with sqlite3.connect("tienda.db") as conn:
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.executemany(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?)",
        pedidos_extra,
    )

    cur.execute("SELECT id, email, total, status, created_at FROM pedidos ORDER BY id DESC LIMIT 5;")
    rows = cur.fetchall()

print("Últimos 5 pedidos (acceso por nombre):")
for r in rows:
    print(dict(r))

Últimos 5 pedidos (acceso por nombre):
{'id': 7, 'email': 'luis@example.com', 'total': 120.0, 'status': 'cancelled', 'created_at': '2026-01-26T16:53:27'}
{'id': 6, 'email': 'ana@example.com', 'total': 3.5, 'status': 'paid', 'created_at': '2026-01-26T16:53:27'}
{'id': 5, 'email': 'ana@example.com', 'total': 9.99, 'status': 'paid', 'created_at': '2026-01-26T16:53:27'}
{'id': 4, 'email': 'luis@example.com', 'total': 55.0, 'status': 'paid', 'created_at': '2026-01-26T16:53:05'}
{'id': 3, 'email': 'ana@example.com', 'total': 19.99, 'status': 'paid', 'created_at': '2026-01-26T16:53:05'}


## Modelo relacional realista: claves foráneas y JOINs

En proyectos reales rara vez tienes una sola tabla. Lo típico es modelar relaciones:

- **1:N** (un usuario tiene muchos pedidos)
- **N:M** (un pedido tiene muchos productos y un producto aparece en muchos pedidos)  
  Esto se representa con una tabla intermedia (`pedido_items`).

### JOINs que usaremos
- `INNER JOIN`: devuelve solo filas que tienen correspondencia en ambas tablas.
- `LEFT JOIN`: devuelve todas las filas de la izquierda, aunque no haya correspondencia (muy útil para detectar "usuarios sin pedidos").


In [7]:
import sqlite3
from pathlib import Path
from datetime import datetime

db_rel = Path("tienda_relacional.db")

def now_iso():
    return datetime.now().isoformat(timespec="seconds")

with sqlite3.connect(db_rel) as conn:
    # Importante: en SQLite las claves foráneas hay que activarlas por conexión
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # Limpieza para que el ejemplo sea re-ejecutable
    cur.executescript("""
    DROP TABLE IF EXISTS pedido_items;
    DROP TABLE IF EXISTS pedidos;
    DROP TABLE IF EXISTS productos;
    DROP TABLE IF EXISTS usuarios;
    """)

    # Tabla usuarios (email como PK: ejemplo didáctico)
    cur.execute("""
        CREATE TABLE usuarios (
            email TEXT PRIMARY KEY,
            nombre TEXT NOT NULL,
            pais TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
    """)

    # Tabla productos
    cur.execute("""
        CREATE TABLE productos (
            sku TEXT PRIMARY KEY,
            nombre TEXT NOT NULL,
            precio REAL NOT NULL CHECK(precio >= 0)
        );
    """)

    # Tabla pedidos: FK a usuarios
    cur.execute("""
        CREATE TABLE pedidos (
            id INTEGER PRIMARY KEY,
            user_email TEXT NOT NULL,
            status TEXT NOT NULL CHECK(status IN ('paid', 'pending', 'cancelled')),
            created_at TEXT NOT NULL,
            FOREIGN KEY (user_email) REFERENCES usuarios(email)
                ON UPDATE CASCADE
                ON DELETE RESTRICT
        );
    """)

    # Tabla intermedia N:M (pedido <-> productos)
    cur.execute("""
        CREATE TABLE pedido_items (
            pedido_id INTEGER NOT NULL,
            sku TEXT NOT NULL,
            qty INTEGER NOT NULL CHECK(qty > 0),
            price_at_purchase REAL NOT NULL CHECK(price_at_purchase >= 0),
            PRIMARY KEY (pedido_id, sku),
            FOREIGN KEY (pedido_id) REFERENCES pedidos(id) ON DELETE CASCADE,
            FOREIGN KEY (sku) REFERENCES productos(sku) ON DELETE RESTRICT
        );
    """)

    # Datos de ejemplo
    usuarios = [
        ("ana@example.com", "Ana", "ES", now_iso()),
        ("luis@example.com", "Luis", "ES", now_iso()),
        ("mia@example.com", "Mia", "PT", now_iso()),
    ]
    productos = [
        ("BOOK-001", "Libro Python", 30.0),
        ("COFFEE-002", "Café", 2.5),
        ("USB-003", "Pendrive 64GB", 12.0),
    ]

    cur.executemany("INSERT INTO usuarios(email, nombre, pais, created_at) VALUES (?, ?, ?, ?);", usuarios)
    cur.executemany("INSERT INTO productos(sku, nombre, precio) VALUES (?, ?, ?);", productos)

    # Pedidos (guardamos IDs generados)
    cur.execute("INSERT INTO pedidos(user_email, status, created_at) VALUES (?, ?, ?);", ("ana@example.com", "paid", now_iso()))
    pedido_ana_1 = cur.lastrowid
    cur.execute("INSERT INTO pedidos(user_email, status, created_at) VALUES (?, ?, ?);", ("ana@example.com", "pending", now_iso()))
    pedido_ana_2 = cur.lastrowid
    cur.execute("INSERT INTO pedidos(user_email, status, created_at) VALUES (?, ?, ?);", ("luis@example.com", "paid", now_iso()))
    pedido_luis_1 = cur.lastrowid

    items = [
        (pedido_ana_1, "BOOK-001", 1, 30.0),
        (pedido_ana_1, "COFFEE-002", 2, 2.5),
        (pedido_ana_2, "USB-003", 1, 12.0),
        (pedido_luis_1, "COFFEE-002", 5, 2.5),
    ]
    cur.executemany(
        "INSERT INTO pedido_items(pedido_id, sku, qty, price_at_purchase) VALUES (?, ?, ?, ?);",
        items
    )

print("DB relacional creada:", db_rel)


DB relacional creada: tienda_relacional.db


In [8]:
import sqlite3

db_rel = "tienda_relacional.db"

with sqlite3.connect(db_rel) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.row_factory = sqlite3.Row

    print("Pedidos con usuario (INNER JOIN):")
    q1 = """
    SELECT p.id, u.email, u.nombre, p.status, p.created_at
    FROM pedidos p
    INNER JOIN usuarios u ON u.email = p.user_email
    ORDER BY p.id;
    """
    for r in conn.execute(q1):
        print(dict(r))

    print("\nLíneas de pedido con producto (JOIN múltiple):")
    q2 = """
    SELECT p.id AS pedido_id, u.email, pr.sku, pr.nombre AS producto, i.qty, i.price_at_purchase,
           (i.qty * i.price_at_purchase) AS subtotal
    FROM pedido_items i
    JOIN pedidos p   ON p.id = i.pedido_id
    JOIN usuarios u  ON u.email = p.user_email
    JOIN productos pr ON pr.sku = i.sku
    ORDER BY p.id, pr.sku;
    """
    for r in conn.execute(q2):
        print(dict(r))

    print("\nTotal por pedido (GROUP BY):")
    q3 = """
    SELECT p.id AS pedido_id, u.email,
           SUM(i.qty * i.price_at_purchase) AS total
    FROM pedidos p
    JOIN usuarios u ON u.email = p.user_email
    JOIN pedido_items i ON i.pedido_id = p.id
    GROUP BY p.id, u.email
    ORDER BY total DESC;
    """
    for r in conn.execute(q3):
        print(dict(r))

    print("\nUsuarios sin pedidos (LEFT JOIN):")
    q4 = """
    SELECT u.email, u.nombre, COUNT(p.id) AS n_pedidos
    FROM usuarios u
    LEFT JOIN pedidos p ON p.user_email = u.email
    GROUP BY u.email, u.nombre
    HAVING n_pedidos = 0;
    """
    rows = list(conn.execute(q4))
    if rows:
        for r in rows:
            print(dict(r))
    else:
        print("Ninguno")


Pedidos con usuario (INNER JOIN):
{'id': 1, 'email': 'ana@example.com', 'nombre': 'Ana', 'status': 'paid', 'created_at': '2026-01-26T16:57:40'}
{'id': 2, 'email': 'ana@example.com', 'nombre': 'Ana', 'status': 'pending', 'created_at': '2026-01-26T16:57:40'}
{'id': 3, 'email': 'luis@example.com', 'nombre': 'Luis', 'status': 'paid', 'created_at': '2026-01-26T16:57:40'}

Líneas de pedido con producto (JOIN múltiple):
{'pedido_id': 1, 'email': 'ana@example.com', 'sku': 'BOOK-001', 'producto': 'Libro Python', 'qty': 1, 'price_at_purchase': 30.0, 'subtotal': 30.0}
{'pedido_id': 1, 'email': 'ana@example.com', 'sku': 'COFFEE-002', 'producto': 'Café', 'qty': 2, 'price_at_purchase': 2.5, 'subtotal': 5.0}
{'pedido_id': 2, 'email': 'ana@example.com', 'sku': 'USB-003', 'producto': 'Pendrive 64GB', 'qty': 1, 'price_at_purchase': 12.0, 'subtotal': 12.0}
{'pedido_id': 3, 'email': 'luis@example.com', 'sku': 'COFFEE-002', 'producto': 'Café', 'qty': 5, 'price_at_purchase': 2.5, 'subtotal': 12.5}

Total po

## Control de excepciones (SQLite) y transacciones

En producción, lo normal es manejar errores de BD:
- `sqlite3.IntegrityError`: violación de `UNIQUE`, `PRIMARY KEY`, `CHECK`, `FOREIGN KEY`
- `sqlite3.OperationalError`: SQL mal escrito, tabla inexistente, etc.
- `sqlite3.Error`: clase base (para capturar "cualquier error SQLite")

Con `with sqlite3.connect(...) as conn:`:
- si todo va bien → **commit**
- si hay excepción → **rollback automático**


In [9]:
import sqlite3

db_rel = "tienda_relacional.db"

# 1) Ejemplo de UNIQUE/PRIMARY KEY: SKU duplicado
try:
    with sqlite3.connect(db_rel) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")
        conn.execute("INSERT INTO productos(sku, nombre, precio) VALUES (?, ?, ?);", ("BOOK-001", "Duplicado", 1.0))
except sqlite3.IntegrityError as e:
    print("IntegrityError (duplicado):", e)

# 2) Ejemplo de FOREIGN KEY: item con SKU que no existe
try:
    with sqlite3.connect(db_rel) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")
        conn.execute(
            "INSERT INTO pedido_items(pedido_id, sku, qty, price_at_purchase) VALUES (?, ?, ?, ?);",
            (1, "NO-EXISTE", 1, 9.99),
        )
except sqlite3.IntegrityError as e:
    print("IntegrityError (FK):", e)

# 3) Ejemplo de error de SQL (tabla mal)
try:
    with sqlite3.connect(db_rel) as conn:
        conn.execute("SELECT * FROM tabla_que_no_existe;")
except sqlite3.OperationalError as e:
    print("OperationalError:", e)


IntegrityError (duplicado): UNIQUE constraint failed: productos.sku
IntegrityError (FK): FOREIGN KEY constraint failed
OperationalError: no such table: tabla_que_no_existe


## Transacciones e índices

- Una transacción agrupa cambios: o se aplican todos o ninguno.
- Los índices aceleran `WHERE`/`JOIN`, pero ralentizan inserciones (ligeramente).

Aquí indexamos `email` y `status`, típicos campos de consulta.

In [10]:
import sqlite3

with sqlite3.connect("tienda.db") as conn:
    cur = conn.cursor()
    cur.execute("CREATE INDEX IF NOT EXISTS idx_pedidos_email ON pedidos(email)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_pedidos_status ON pedidos(status)")
print("Índices creados")

Índices creados


## Índices y `EXPLAIN QUERY PLAN` (mirar qué está haciendo SQLite)

Un índice acelera búsquedas y joins por columnas frecuentes (`WHERE`, `JOIN`, `ORDER BY`).
Puedes ver el plan aproximado con `EXPLAIN QUERY PLAN ...`.


In [11]:
import sqlite3

db_rel = "tienda_relacional.db"

with sqlite3.connect(db_rel) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    query = "SELECT * FROM pedidos WHERE user_email = ? AND status = ?;"
    params = ("ana@example.com", "paid")

    print("Plan ANTES de índices:")
    for r in cur.execute("EXPLAIN QUERY PLAN " + query, params):
        print(r)

    cur.execute("CREATE INDEX IF NOT EXISTS idx_pedidos_user_status ON pedidos(user_email, status);")

    print("\nPlan DESPUÉS de índices:")
    for r in cur.execute("EXPLAIN QUERY PLAN " + query, params):
        print(r)

    print("\nEjecutando consulta:")
    for r in cur.execute(query, params):
        print(r)


Plan ANTES de índices:
(2, 0, 216, 'SCAN pedidos')

Plan DESPUÉS de índices:
(3, 0, 62, 'SEARCH pedidos USING INDEX idx_pedidos_user_status (user_email=? AND status=?)')

Ejecutando consulta:
(1, 'ana@example.com', 'paid', '2026-01-26T16:57:40')


## Integración con Pandas

Pandas puede leer directamente SQL, útil para análisis y reporting.

Nota: esto requiere tener `pandas` instalado en tu entorno.

In [12]:
import sqlite3

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("Pandas no está instalado en este entorno. Instálalo para usar read_sql_query.")
else:
    with sqlite3.connect("tienda.db") as conn:
        df = pd.read_sql_query("SELECT * FROM pedidos", conn)
    display(df)

,id,email,total,status,created_at
0,1,ana@example.com,19.99,paid,2026-01-26T16:51:11
1,2,luis@example.com,55.00,paid,2026-01-26T16:51:11
2,3,ana@example.com,19.99,paid,2026-01-26T16:53:05
3,4,luis@example.com,55.00,paid,2026-01-26T16:53:05
4,5,ana@example.com,9.99,paid,2026-01-26T16:53:27
5,6,ana@example.com,3.50,paid,2026-01-26T16:53:27
6,7,luis@example.com,120.00,cancelled,2026-01-26T16:53:27


## Trampas comunes y buenas prácticas

- Usa `with sqlite3.connect(...) as conn:` para **commit/rollback automático**.
- Parametriza SIEMPRE (`?`) los datos externos. Nada de f-strings en SQL.
- Activa claves foráneas por conexión cuando las uses: `conn.execute("PRAGMA foreign_keys = ON;")`.
- Elige un formato de fecha/hora y sé consistente (en este módulo: **ISO-8601 en TEXT**).
- Para insertar lotes: `executemany(...)` (más rápido y más limpio).
- Para resultados legibles: `conn.row_factory = sqlite3.Row` y `dict(row)`.
- Añade índices solo donde tenga sentido: columnas usadas en `WHERE`, `JOIN`, `ORDER BY`. Comprueba con `EXPLAIN QUERY PLAN`.
- Evita “base de datos como fichero compartido” con muchas escrituras concurrentes; SQLite brilla en 1 proceso o pocas escrituras.
- Si algo falla, captura `sqlite3.IntegrityError` y `sqlite3.OperationalError` para dar mensajes útiles.


## Mini-práctica

1) Crea una tabla `usuarios(email PRIMARY KEY, pais, created_at)`.  
2) Inserta 20 usuarios (puedes generarlos).  
3) Crea un índice por `pais`.  
4) Consulta cuántos usuarios hay por país (`GROUP BY`).  
5) Lee el resultado con Pandas (si lo tienes instalado).

**Bonus (relacional):**
6) Crea una tabla `pedidos(id INTEGER PRIMARY KEY, user_email TEXT, total REAL, created_at TEXT)` con `FOREIGN KEY(user_email) REFERENCES usuarios(email)`.  
7) Inserta pedidos para algunos usuarios y deja otros sin pedidos.  
8) Haz un `LEFT JOIN` para listar usuarios y número de pedidos (incluyendo los que tienen 0).  
